In [1]:
from imutils import contours
import numpy as np
import argparse
import imutils
import cv2
import myutils

In [2]:
# 指定信用卡的类型
FISRT_NUMBER = {
    "3": "American Express",
    "4": "Visa",
    "5": "MasterCard",
    "6": "Discover Card" 
}

In [29]:
def sort_contours(cnts, method="left-to-right"):
    reverse = False
    i = 0

    if method == "right-to-left" or method == "bottom-to-top":
        reverse = True
    if method == "top-to-bottom" or method == "left-to-right":
        i = 1
    boundingBoxes = [cv2.boundingRect(c) for c in cnts]
    (cnts, boundingBoxes) = zip(*sorted(zip(cnts, boundingBoxes), key=lambda b: b[1][i], reverse=reverse))

    return cnts, boundingBoxes

In [30]:
def resize(image, width=None, height=None, inter=cv2.INTER_AREA):
    dim = None
    (h,w) = image.shape[:2]
    if width is None and height is None:
        return image
    if width is None:
        r = height / float(h)
        dim = (int(w*r), height)
    else:
        r = width / float(w)
        dim = (width, int(h*r))
    resized = cv2.resize(image, dim, interpolation=inter)
    return resized

In [3]:
def cv_show(img, name):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [24]:
# 读取模板图像
img = cv2.imread('images/ocr_a_reference.png')
ref = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [25]:
# 转换为二值图像
ref = cv2.threshold(ref, 10, 255, cv2.THRESH_BINARY_INV)[1]
cv_show(img_B, 'BINARY_INV')

In [26]:
# 计算轮廓
# cv2.findContours() 函数接受的参数为二值图（黑白图）,cv2.RETR_EXTERNAL只检测外轮廓，cv2.CHAIN_APPROX_SIMPLE只保留终点坐标
# 返回的list中，每个元素都是图像的一个轮廓

ref_, refCnts, hierarchy = cv2.findContours(ref.copy(), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)

cv2.drawContours(img, refCnts, -1, (0,0,255), 1)
cv_show(img, "img")

In [27]:
print(np.array(refCnts).shape)

(15,)


/home/li325/miniconda3/envs/opencv34/lib/python3.7/site-packages/ipykernel_launcher.py:1: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  """Entry point for launching an IPython kernel.


In [31]:
refCnts = sort_contours(refCnts, method="left-to-right")[0]
digits = {}

for (i, c) in enumerate(refCnts):
    # 计算外界矩形并且 resize 成合适的大小
    (x, y, h, w) = cv2.boundingRect(c)
    roi = ref[y:y+h, x:x+w]
    roi = cv2.resize(roi, (57,88))

    # 每一个数字对应一个模板
    digits[i] = roi

In [32]:
# 初始化卷几何
rectKernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9,3))
sqKernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))

In [34]:
# 读取输入图像，预处理
image = cv2.imread("images/credit_card_03.png")
cv_show(image, 'image')
image = resize(image, width=300)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
cv_show(gray, 'gray')

In [36]:
# 礼帽操作，突出更明亮的区域
tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, rectKernel)
cv_show(tophat, 'tophat')

In [37]:
gradX = cv2.Sobel(tophat, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=-1) # ksize=-1相当于用3*3卷几何

gradX = np.absolute(gradX)
(minVal, maxVal) = (np.min(gradX), np.max(gradX))
gradX = (255 * ((gradX-minVal) / (maxVal-gradX)))
gradX = gradX.astype("uint8")

print(np.array(gradX).shape)
cv_show(gradX, 'gradX')

/home/li325/miniconda3/envs/opencv34/lib/python3.7/site-packages/ipykernel_launcher.py:5: RuntimeWarning: divide by zero encountered in true_divide
  """


(194, 300)


In [38]:
# 通过闭操作（先膨胀，再腐蚀）将数字连在一起
gradX = cv2.morphologyEx(gradX, cv2.MORPH_CLOSE, rectKernel)
cv_show(gradX, "gradX")

In [39]:
# THRESH_OTSU 会自动寻找合适的阈值，适合双峰。需要把阈值参数设为0
thresh = cv2.threshold(gradX, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
cv_show(thresh, 'thresh')

In [ ]:
# 再来一个闭操作
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, sqKernel)
cv_show(thresh,'thresh')

In [47]:
# 计算轮廓
thresh_, threshCnts, hierarchy = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

cnts = threshCnts
cur_img = image.copy()
cv2.drawContours(cur_img, cnts, -1, (0,0,255), 3)
cv_show(cur_img, 'cur_img')

In [55]:
locs = []

# 遍历轮廓
for (i,c) in enumerate(cnts):
    (x, y, w, h) = cv2.boundingRect(c)
    ar = w / float(h)

    # 选择合适区域
    if ar>2.5 and ar<4.0:
        if (w>40 and w<55) and (h>10 and h<20):
            locs.append((x,y,w,h))

# 将符合的轮廓从左到右排序
locs = sorted(locs, key=lambda x:x[0])
output = []
print("定位到的数字分组数量：", len(locs))

定位到的数字分组数量： 4


In [56]:
# 遍历每一个轮廓中的数字
for (i, (gx,gy,gw,gh)) in enumerate(locs):
    groupOutput = []

    # 根据坐标提取每一个组
    group = gray[gy-5:gy+gh+5, gx-5:gx+gw+5]
    cv_show(group, 'group')

    # 预处理
    group = cv2.threshold(group, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    cv_show(group, 'group')

    # 计算每一组的轮廓
    group_, digitCnts, hierarchy = cv2.findContours(group.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    digitCnts = contours.sort_contours(digitCnts, method="left-to-right")[0]

    # 计算每一组中的每一个数值
    for c in digitCnts:
        # 找到当前数值的轮廓，resize 成合适的大小
        (x,y,w,h) = cv2.boundingRect(c)
        roi = group[y:y+h, x:x+w]
        roi = cv2.resize(roi, (57, 88))
        cv_show(roi, 'roi')

        # 计算匹配得分
        scores = []

        # 在模板中计算每一个的得分
        for (digit, digitROI) in digits.items():
            # 模板匹配
            result = cv2.matchTemplate(roi, digitROI, cv2.TM_CCOEFF)
            (_, score, _, _) = cv2.minMaxLoc(result)
            scores.append(score)
        
        # 得到最合适的数字
        groupOutput.append(str(np.argmax(scores)))
    
    # 画出来
    cv2.rectangle(image, (gx-5, gy-5), (gx+gw+5, gy+gh+5), (0,0,255), 1)
    cv2.putText(image, "".join(groupOutput), (gx,gy-15), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,255), 2)

    # 得到结果
    output.extend(groupOutput)

print(len(output))

16


In [57]:
print("Credit Card Type: {}".format(FISRT_NUMBER[output[0]]))
print("Credit Card #: {}".format("".join(output)))
cv2.imshow("Image", image)
cv2.waitKey(0)

KeyError: '13'